## Script to filter (copy) the val/test split images of a COCO dataset into a new mirrored directory

Reads the annotation JSON(s) of the splits you want to keep (e.g. validation + test),
collects every listed image, and **copies** those files into a new dataset directory
that mirrors the original folder structure. The source dataset is **never modified**:
files are only opened in read-only mode, nothing is moved or deleted.

In [ ]:
import os
import json
import shutil
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv


def _load_dotenv():
    """Load .env from the current directory upward (repo root); no-op if not found."""
    d = os.path.abspath(os.getcwd())
    while True:
        env_path = os.path.join(d, ".env")
        if os.path.isfile(env_path):
            return load_dotenv(env_path)
        parent = os.path.dirname(d)
        if parent == d:
            return False
        d = parent


_load_dotenv()  # reads ZERO_SHOT_DATA_DIR (and HF_TOKEN) from .env

DATASETS_PATH = os.environ.get("ZERO_SHOT_DATA_DIR", "./data")

# One entry per dataset. For each dataset:
#   dataset_dir : root of the original dataset (must NOT be nested inside output_dir)
#   images_dir  : optional subfolder holding the images (defaults to dataset_dir itself)
#   splits      : annotation JSONs whose images should be KEPT (typically val + test);
#                 'images_dir' can be overridden per split when a split's images live in
#                 a different physical folder (e.g. ecostack)
#   output_dir  : NEW dataset root, created if missing; nothing here is ever written to
#                 the original dataset
DATASETS = {
    "pitfall-cameras": {
        "dataset_dir": f"{DATASETS_PATH}/pitfall-cameras",
        "images_dir": "images",
        "splits": [
            {"name": "val", "json": f"{DATASETS_PATH}/pitfall-cameras/pitfall-cameras_val_unified_SAM3.json"},
            {"name": "test", "json": f"{DATASETS_PATH}/pitfall-cameras/pitfall-cameras_test_unified.json"},
        ],
        "output_dir": f"{DATASETS_PATH}/pitfall-cameras_val_test",
    },
    "ecostack": {
        "dataset_dir": f"{DATASETS_PATH}/ecostack",
        "splits": [
            {"name": "val", "json": f"{DATASETS_PATH}/ecostack/val1201_coco_unified.json", "images_dir": "val1201"},
            {"name": "test", "json": f"{DATASETS_PATH}/ecostack/test1201_coco_unified.json", "images_dir": "test1201"},
        ],
        "output_dir": f"{DATASETS_PATH}/ecostack_val_test",
    },
    "stefan_smartphone": {
        "dataset_dir": f"{DATASETS_PATH}/stefan_smartphone",
        "images_dir": "cropped",
        "splits": [
            {"name": "val", "json": f"{DATASETS_PATH}/stefan_smartphone/detections_val_20.json"},
            {"name": "test", "json": f"{DATASETS_PATH}/stefan_smartphone/detections_test_80.json"},
        ],
        "output_dir": f"{DATASETS_PATH}/stefan_smartphone_val_test",
    },
    "rangex": {
        "dataset_dir": f"{DATASETS_PATH}/rangex",
        "splits": [
            {"name": "val", "json": f"{DATASETS_PATH}/rangex/detections_val_20_scene.json"},
            {"name": "test", "json": f"{DATASETS_PATH}/rangex/detections_test_80_scene.json"},
        ],
        "output_dir": f"{DATASETS_PATH}/rangex_val_test",
    },
}

DATASET = "pitfall-cameras"  # selector: switch to run another dataset"""

In [ ]:
def load_split_images(json_path):
    """Return the 'images' list of a COCO annotation JSON (entries have id, file_name, ...)."""
    with open(json_path) as f:
        return json.load(f)["images"]


def resolve_image_path(img, split_cfg, cfg):
    """Locate the source file of an image on disk, trying several candidate paths."""
    images_dir = split_cfg.get("images_dir") or cfg.get("images_dir") or "."
    candidates = [
        os.path.join(cfg["dataset_dir"], images_dir, img["file_name"]),
        os.path.join(cfg["dataset_dir"], img["file_name"]),
        img["file_name"],  # in case the JSON stores an absolute path
    ]
    return next((p for p in candidates if os.path.isfile(p)), candidates[0])


def mirror_split_images(split_cfg, cfg, copied_sources, dest_to_src):
    """Copy every image listed in one split JSON into the mirrored output directory."""
    images = load_split_images(split_cfg["json"])
    stats = {"split": split_cfg["name"], "json_images": len(images),
             "copied": 0, "dups": 0, "missing": 0, "collisions": 0, "missing_files": []}

    for img in tqdm(images, desc=f"[{split_cfg['name']}] copying"):
        source = resolve_image_path(img, split_cfg, cfg)
        if not os.path.isfile(source):
            stats["missing"] += 1
            if len(stats["missing_files"]) < 5:
                stats["missing_files"].append(img["file_name"])
            continue

        # Dedup on the resolved source path (NOT on file_name: ecostack's val/test
        # share the same file names but live in different physical folders).
        if os.path.abspath(source) in copied_sources:
            stats["dups"] += 1
            continue
        copied_sources.add(os.path.abspath(source))

        # Mirror the original folder structure: dest = output_dir/<relative path under dataset_dir>
        rel = os.path.relpath(source, cfg["dataset_dir"])
        if rel.startswith(".."):
            # Source outside the dataset root (absolute file_name): flatten, never escape output_dir
            rel = os.path.basename(img["file_name"])
        dest = os.path.join(cfg["output_dir"], rel)

        if dest in dest_to_src and dest_to_src[dest] != source:
            stats["collisions"] += 1
            print(f"  COLLISION: {source} -> {dest} (already {dest_to_src[dest]}); skipping")
            continue
        dest_to_src[dest] = source

        os.makedirs(os.path.dirname(dest), exist_ok=True)
        shutil.copy2(source, dest)
        stats["copied"] += 1

    return stats


def _tree_snapshot(root):
    """Count files and total bytes under *root* (used for the source-untouched check)."""
    n_files, total_bytes = 0, 0
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            n_files += 1
            total_bytes += os.path.getsize(os.path.join(dirpath, name))
    return n_files, total_bytes


def mirror_dataset(cfg):
    """Copy the images of all configured splits into cfg['output_dir'], mirroring structure.

    Returns a dict with:
      stats          : per-split summary DataFrame
      snapshot_before: (file_count, total_bytes) of the source tree before copying
      dest_to_src    : {dest_path: source_path} for every copied file
    """
    snapshot_before = _tree_snapshot(cfg["dataset_dir"])
    copied_sources, dest_to_src = set(), {}
    rows = [mirror_split_images(split_cfg, cfg, copied_sources, dest_to_src)
            for split_cfg in cfg["splits"]]

    stats = pd.DataFrame(rows)
    print(f"\nImages: {stats['json_images'].sum()}, Copied: {stats['copied'].sum()}, "
          f"Duplicates: {stats['dups'].sum()}, Missing: {stats['missing'].sum()}, "
          f"Collisions: {stats['collisions'].sum()}, Saved to: {cfg['output_dir']}")
    return {"stats": stats, "snapshot_before": snapshot_before, "dest_to_src": dest_to_src}

In [ ]:
cfg = DATASETS[DATASET]
result = mirror_dataset(cfg)
stats = result["stats"]

In [ ]:
stats

In [ ]:
dests = sorted(result["dest_to_src"])
print(f"Mirrored structure preview for '{DATASET}' ({len(dests)} files in {cfg['output_dir']}):")
for dest in dests[:10]:
    print(f"  {os.path.relpath(dest, cfg['output_dir'])}")
if len(dests) > 10:
    print(f"  ... and {len(dests) - 10} more")

In [ ]:
# The original dataset must be untouched: file count and total bytes are unchanged,
# and every copied file is byte-identical (same size) to its source.
n_files, total_bytes = _tree_snapshot(cfg["dataset_dir"])
assert (n_files, total_bytes) == result["snapshot_before"], "Source dataset changed during filtering!"
for dest, source in result["dest_to_src"].items():
    assert os.path.getsize(dest) == os.path.getsize(source), f"Size mismatch: {dest}"
print(f"Source dataset untouched: OK ({n_files} files, {total_bytes} bytes; {len(result['dest_to_src'])} identical copies)")